# MapBiomas Fire — GCS Export Validation

Verifies if the **chunks (M1)** and **COGs (M2)** for MapBiomas Fire imagery catalogs have been correctly exported to Google Cloud Storage (GCS).

Supports all South American countries, satellite sensors (**Sentinel-2**, **Landsat**), and temporal resolutions (**Annual**, **Monthly**).

## Validated Structure

| Field | Description / Dynamic Value |
|-------|-----------------------------|
| Bucket | `mapbiomas-fire` |
| Root Path | `sudamerica/{country}/CATALOG_01/LIBRARY_IMAGES/{sensor}/{period}/` |
| Mosaic Methods | `MINNBR`, `MINNBR_BUFFER` |
| Standard Bands | `blue`, `green`, `red`, `nir`, `swir1`, `swir2`, `dayOfYear` |

**Zero download constraint**: Performs lightweight metadata inspection only (file availability, chunk counts, file sizes).

In [ ]:
# @title 1. Setup — Environment and Authentication
import os, sys, json, datetime, time, re, textwrap, warnings
warnings.filterwarnings('ignore')
from collections import defaultdict

IS_COLAB = 'COLAB_RELEASE_TAG' in os.environ
print(f'Environment: {"Google Colab" if IS_COLAB else "Local Workspace"}')

if IS_COLAB:
    !pip install -q gcsfs pandas tabulate 2>&1 | tail -1
    from google.colab import auth
    auth.authenticate_user()
    print('GCP authenticated via Google Colab.')
else:
    print('Assuming local Application Default Credentials (ADC).')

import gcsfs
import pandas as pd

fs = gcsfs.GCSFileSystem()
print('gcsfs client initialized successfully.')

In [ ]:
# @title 2. Configuration Parameters
# Target bucket and catalog parameters
BUCKET = 'mapbiomas-fire'
COUNTRY = 'bolivia'  # e.g., 'peru', 'bolivia', 'brasil', 'colombia', 'paraguay'
SENSOR = 'LANDSAT'   # ['SENTINEL2', 'LANDSAT']
PERIOD = 'ANNUALLY'  # ['ANNUALLY', 'MONTHLY']
MOSAIC_METHODS = ['MINNBR']  # ['MINNBR', 'MINNBR_BUFFER']

# Base GCS path construction
GCS_BASE = f'{BUCKET}/sudamerica/{COUNTRY.lower()}/CATALOG_01/LIBRARY_IMAGES/{SENSOR.upper()}/{PERIOD.upper()}'
BANDS = ['blue', 'green', 'red', 'nir', 'swir1', 'swir2', 'dayOfYear']

# Dynamic naming generator for exported mosaic files
def mosaic_name(year, month_or_none, band, mosaic):
    m = mosaic.lower()
    c = COUNTRY.lower()
    s = SENSOR.lower()
    if PERIOD == 'ANNUALLY':
        return f'image_{c}_fire_{s}_{m}_{band}_{year}'
    else:
        return f'image_{c}_fire_{s}_{m}_{band}_{year}_{month_or_none:02d}'

print(f'Target Bucket: {BUCKET}')
print(f'Base GCS Path: {GCS_BASE}')
print(f'Selected Methods: {MOSAIC_METHODS}')
print(f'Monitored Bands: {BANDS}')

In [ ]:
# @title 3. 🔎 GCS Scan
def scan_gcs(mosaic_methods=None, bands=None):
    rows, total_periods = [], 0
    for mosaic in (mosaic_methods or MOSAIC_METHODS):
        base = f'{GCS_BASE}/{mosaic}'
        try:
            pattern = r'^\d{4}$' if PERIOD == 'ANNUALLY' else r'^\d{4}_\d{2}$'
            periods = sorted(e.rstrip('/').split('/')[-1] for e in fs.ls(base) if fs.isdir(e) and re.match(pattern, e.rstrip('/').split('/')[-1]))
        except Exception as e:
            print(f'⚠️ {mosaic}: {e}')
            continue

        total_periods += len(periods)
        print(f'📁 [{COUNTRY.upper()} | {SENSOR} | {mosaic}]: {len(periods)} periods found ({PERIOD})')
        for p in periods:
            files = {}
            for kind in ['CHUNKS', 'COG']:
                try:
                    files[kind] = [f for f in fs.find(f'{base}/{p}/{kind}') if f.endswith('.tif')]
                except FileNotFoundError:
                    files[kind] = []
                except Exception as e:
                    print(f'⚠️ {p}/{kind}: {e}')
                    files[kind] = []

            for band in (bands or BANDS):
                b = band.lower()
                match = lambda f: f'_{b}_' in f.lower() or f'_{b}.' in f.lower()
                chunks = [f for f in files['CHUNKS'] if match(f)]
                cogs = [f for f in files['COG'] if match(f)]
                sizes = []
                for f in chunks:
                    try: sizes.append(fs.info(f)['size'])
                    except Exception: pass
                try: cog_size = fs.info(cogs[0])['size'] if cogs else 0
                except Exception: cog_size = 0

                rows.append({
                    'mosaic': mosaic, 'period_folder': p, 'year': int(p[:4]),
                    'month': int(p[5:7]) if PERIOD == 'MONTHLY' else None,
                    'band': band, 'n_chunks': len(chunks),
                    'chunks_bytes': sum(sizes), 'chunks_min': min(sizes, default=0),
                    'chunks_max': max(sizes, default=0), 'n_cogs': len(cogs),
                    'has_cog': bool(cogs), 'cog_bytes': cog_size
                })
    return rows, total_periods

print(f'🔎 Starting GCS scan for {COUNTRY.upper()} ({SENSOR})...')
t0 = time.time()
records, n_months = scan_gcs()
print(f'✅ Completed in {time.time()-t0:.1f}s | 📊 {len(records)} records | 📅 {n_months} periods')

In [ ]:
# @title 4. ✅ Validation, Anomalies, and Completeness
import datetime
import pandas as pd

df = pd.DataFrame(records)
if df.empty:
    print(f'⚠️ No data found in GCS for {COUNTRY.upper()} | {SENSOR} | {PERIOD}.')
else:
    df['status'] = df.apply(lambda r: 'MISSING' if r['n_chunks'] == 0 else 'OK' if r['has_cog'] else 'CHUNKS_ONLY', axis=1)

    # 1. Simplified Summary by Period
    print('📅 SUMMARY BY PERIOD')
    summary = df.groupby(['mosaic', 'period_folder', 'status']).size().unstack(fill_value=0)
    for mosaic, sub in summary.groupby(level=0):
        print(f'\n🛰️ {mosaic}')
        for (_, period), row in sub.sort_index(ascending=False).iterrows():
            ok, only, miss = row.get('OK', 0), row.get('CHUNKS_ONLY', 0), row.get('MISSING', 0)
            total = ok + only + miss
            icon = '✅' if ok == total else '⚠️' if only else '❌'
            
            parts = [f'{ok}/{total} COGs OK']
            if only > 0:
                parts.append(f'⚠️ {only} chunks only')
            if miss > 0:
                parts.append(f'❌ {miss} missing')
            
            print(f'  {icon} {period}: ' + ' | '.join(parts))

    # 2. Anomalies
    print('\n🔎 ANOMALIES DETECTED')
    anomalies = df[df['status'] != 'OK']
    if anomalies.empty:
        print('✅ No anomalies detected.')
    else:
        for r in anomalies.itertuples():
            icon = '⚠️' if r.status == 'CHUNKS_ONLY' else '❌'
            detail = f'{r.n_chunks} chunks without COG' if r.status == 'CHUNKS_ONLY' else 'Missing band'
            print(f'  {icon} [{r.mosaic}] {r.period_folder} / {r.band}: {detail}')
        print(f'📌 Total: {len(anomalies)} anomalies')

    # 3. Accurate One-line Yearly Completeness Summary
    print('\n📆 YEARLY COMPLETENESS')
    today = datetime.date.today()
    mosaics_to_check = MOSAIC_METHODS if df.empty else df['mosaic'].unique()
    
    for mosaic in mosaics_to_check:
        df_m = df[df['mosaic'] == mosaic] if not df.empty else pd.DataFrame()
        complete_years = []
        pending_years = []
        
        # Se for LANDSAT, o ano inicial padrão é 1985; se for SENTINEL2, usa o menor ano encontrado no GCS
        if SENSOR.upper() == 'LANDSAT':
            min_yr = 1985 if df_m.empty else min(1985, df_m['year'].min())
        else:
            min_yr = df_m['year'].min() if not df_m.empty else today.year
            
        max_yr = today.year if df_m.empty else min(df_m['year'].max(), today.year)
        expected_bands = set(BANDS)
        
        for year in range(min_yr, max_yr + 1):
            df_yr = df_m[df_m['year'] == year] if not df_m.empty else pd.DataFrame()
            
            if PERIOD == 'ANNUALLY':
                expected_periods = [str(year)]
            else:
                n = 12 if year < today.year else max(0, today.month - 1)
                expected_periods = [f'{year}_{m:02d}' for m in range(1, n + 1)]
            
            reasons = []
            existing_periods = set(df_yr['period_folder'].unique()) if not df_yr.empty else set()
            missing_periods = set(expected_periods) - existing_periods
            
            # Se a pasta do período não existir
            if missing_periods:
                if PERIOD == 'ANNUALLY':
                    reasons.append('no data folder')
                else:
                    reasons.append(f'missing folder: {", ".join(sorted(missing_periods))}')
            
            # Se a pasta existir, verifica o status das bandas
            for p in expected_periods:
                if p in existing_periods:
                    df_p = df_yr[df_yr['period_folder'] == p]
                    
                    for _, r in df_p[df_p['status'] != 'OK'].iterrows():
                        reasons.append(f'{r["band"]}: {r["status"]}')
                    
                    missing_b = expected_bands - set(df_p['band'])
                    if missing_b:
                        reasons.append(f'missing bands: {", ".join(sorted(missing_b))}')
            
            if not reasons:
                complete_years.append(str(year))
            else:
                pending_years.append(f'{year} ({ "; ".join(reasons) })')
        
        print(f'🛰️ [{mosaic}]')
        if complete_years:
            print(f'  ✅ Complete: {", ".join(complete_years)}')
        if pending_years:
            print(f'  ⚠️ Pending: {"; ".join(pending_years)}')
        if not pending_years:
            print('  🎉 All years complete!')

    # 4. Overall Results Summary
    print('\n' + '─' * 45)
    print(f'🌎 {COUNTRY.upper()} | 🛰️ {SENSOR} | 📅 {PERIOD}')
    print(f'✅ COGs OK: {(df["status"] == "OK").sum()}/{len(df)}')
    print(f'⚠️ Chunks only: {(df["status"] == "CHUNKS_ONLY").sum()}')
    print(f'❌ Missing bands: {(df["status"] == "MISSING").sum()}')
    print('─' * 45)

---
**Validation complete.**

If anomalies were detected, inspect the flagged periods/bands and re-run M1 (chunk export) or M2 (COG mosaicking) accordingly.